# CPP Agent Training v2 — Optimized Overnight Run

**Changes from v1:**
- 10x10 trained **from scratch** (curriculum from 5x5 failed — entropy collapse)
- More timesteps: 5x5=2M, 10x10=5M, 20x20=3M curriculum
- Higher `ent_coef` for larger grids to force exploration
- Lower `learning_rate` for curriculum stages
- `n_envs=16` for better GPU utilization
- **Checkpoints every 500k steps** — never lose progress
- Auto-saves results to Google Drive

Use **Runtime > Change runtime type > T4 GPU**. Expected time: **~6-7h total**.

In [ ]:
# 1. Clone repo + install
!git clone https://github.com/pedrocivita/gym_custom_env-pedrotpc.git
%cd gym_custom_env-pedrotpc
!pip install -q gymnasium numpy stable-baselines3 sb3-contrib pygame-ce torch tensorboard

In [ ]:
# 2. GPU check + CUDA optimizations
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_global_mem / 1e9
    print(f"Memory: {mem_gb:.1f} GB")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# 3. Imports + setup
import os
import json
import time
import numpy as np
import gymnasium as gym
from datetime import datetime
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.logger import configure
from stable_baselines3.common.callbacks import CheckpointCallback, BaseCallback

from gymnasium_env.grid_world_cpp import GridWorldCPPEnv
from utils.feature_extractor import CustomCombinedExtractor

try:
    gym.register(id="gymnasium_env/GridWorldCPP-v0", entry_point=GridWorldCPPEnv)
except Exception:
    pass

os.makedirs("data", exist_ok=True)
os.makedirs("log", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

# Mount Google Drive for auto-save
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/cpp_models'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Drive mounted. Models will auto-save to {DRIVE_DIR}")
except Exception:
    DRIVE_DIR = None
    print("Drive not available. Models saved locally only.")

RESULTS = {}
print("Setup complete.")

In [ ]:
# 4. Helper functions

def get_policy_kwargs():
    return {
        "features_extractor_class": CustomCombinedExtractor,
        "features_extractor_kwargs": {"features_dim": 128},
        "lstm_hidden_size": 128,
        "n_lstm_layers": 1,
        "net_arch": dict(pi=[128, 64], vf=[128, 64]),
        "shared_lstm": False,
        "enable_critic_lstm": True,
    }

def create_env(dim, obstacles, max_steps, n_envs=16):
    env_kwargs = {
        "size": dim,
        "obs_quantity": obstacles,
        "max_steps": max_steps,
        "render_mode": "rgb_array",
    }
    return make_vec_env("gymnasium_env/GridWorldCPP-v0", n_envs=n_envs, env_kwargs=env_kwargs)

def save_to_drive(local_path, name):
    """Copy model to Google Drive for persistence."""
    if DRIVE_DIR:
        import shutil
        src = f"{local_path}.zip"
        if os.path.exists(src):
            shutil.copy2(src, f"{DRIVE_DIR}/{name}.zip")
            print(f"  -> Saved to Drive: {DRIVE_DIR}/{name}.zip")

def test_model(model_path, dim, obstacles, num_episodes=100):
    model = RecurrentPPO.load(model_path, device=DEVICE)
    max_steps = dim * dim * 4
    env = gym.make("gymnasium_env/GridWorldCPP-v0", size=dim, obs_quantity=obstacles,
                   max_steps=max_steps, render_mode="rgb_array")
    full = 0
    coverages = []
    steps_list = []
    for i in range(num_episodes):
        obs, info = env.reset()
        done = truncated = False
        lstm_states = None
        episode_start = np.array([True])
        steps = 0
        while not done and not truncated:
            action, lstm_states = model.predict(obs, state=lstm_states,
                                                episode_start=episode_start, deterministic=True)
            obs, reward, done, truncated, info = env.step(action.item())
            episode_start = np.array([False])
            steps += 1
        coverages.append(info["coverage"])
        steps_list.append(steps)
        if done and not truncated:
            full += 1
    env.close()
    rate = full / num_episodes * 100
    avg_cov = np.mean(coverages) * 100
    avg_steps = np.mean(steps_list)
    print(f"\n{'='*50}")
    print(f"  {dim}x{dim} TEST RESULTS ({num_episodes} episodes)")
    print(f"{'='*50}")
    print(f"  Full Coverage Rate: {rate:.1f}% ({full}/{num_episodes})")
    print(f"  Avg Coverage: {avg_cov:.1f}% (std: {np.std(coverages)*100:.1f}%)")
    print(f"  Avg Steps: {avg_steps:.1f} (std: {np.std(steps_list):.1f})")
    print(f"  Coverage Range: [{np.min(coverages)*100:.1f}%, {np.max(coverages)*100:.1f}%]")
    print(f"{'='*50}\n")
    return {"rate": rate, "avg_cov": avg_cov, "avg_steps": avg_steps,
            "std_cov": np.std(coverages)*100, "std_steps": np.std(steps_list),
            "min_cov": np.min(coverages)*100, "max_cov": np.max(coverages)*100}

class ProgressCallback(BaseCallback):
    """Print concise progress every N steps."""
    def __init__(self, print_freq=50_000, verbose=0):
        super().__init__(verbose)
        self.print_freq = print_freq
        self.start_time = None

    def _on_training_start(self):
        self.start_time = time.time()

    def _on_step(self):
        if self.num_timesteps % self.print_freq < self.training_env.num_envs:
            elapsed = time.time() - self.start_time
            fps = self.num_timesteps / max(elapsed, 1)
            ep_info = self.logger.name_to_value
            rew = ep_info.get('rollout/ep_rew_mean', '?')
            length = ep_info.get('rollout/ep_len_mean', '?')
            if isinstance(rew, float):
                print(f"  [{self.num_timesteps:>9,}] rew={rew:>7.1f} | len={length:>5.0f} | fps={fps:.0f} | {elapsed/60:.1f}min")
        return True

print("Helpers loaded.")

In [ ]:
# 5. STAGE 1: Train 5x5 (fresh, ~40 min)
print("="*60)
print("STAGE 1: Training 5x5 (2M steps)")
print("="*60)

DIM, OBS, MAX_STEPS, TIMESTEPS = 5, 3, 150, 2_000_000

env = create_env(DIM, OBS, MAX_STEPS, n_envs=16)
model = RecurrentPPO(
    "MultiInputLstmPolicy", env, verbose=1,
    learning_rate=3e-4, n_steps=256, batch_size=128, n_epochs=4,
    gamma=0.995, gae_lambda=0.95, ent_coef=0.05,
    clip_range=0.2, max_grad_norm=0.5,
    policy_kwargs=get_policy_kwargs(), device=DEVICE,
)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_dir = f"log/v2_5x5_{ts}"
model.set_logger(configure(log_dir, ["stdout", "csv", "tensorboard"]))

checkpoint_cb = CheckpointCallback(save_freq=250_000 // 16, save_path="checkpoints/",
                                    name_prefix="5x5")

model.learn(total_timesteps=TIMESTEPS, callback=[checkpoint_cb, ProgressCallback()])
model_5x5 = f"data/v2_5x5_{ts}"
model.save(model_5x5)
env.close()
save_to_drive(model_5x5, "v2_5x5_final")
print(f"\n5x5 model saved: {model_5x5}.zip")

In [ ]:
# 6. Test 5x5
RESULTS["5x5"] = test_model(model_5x5, 5, 3)
print(f"Intermediate results so far: {json.dumps({k: {kk: round(vv,1) for kk,vv in v.items()} for k,v in RESULTS.items()}, indent=2)}")

In [ ]:
# 7. STAGE 2: Train 10x10 FROM SCRATCH (5M steps, ~3h)
#    Key change: NOT using curriculum. Training directly on 10x10.
#    Higher ent_coef=0.08 to encourage exploration in larger grid.
print("="*60)
print("STAGE 2: Training 10x10 FROM SCRATCH (5M steps)")
print("="*60)

DIM, OBS, MAX_STEPS, TIMESTEPS = 10, 12, 600, 5_000_000

env = create_env(DIM, OBS, MAX_STEPS, n_envs=16)
model_10 = RecurrentPPO(
    "MultiInputLstmPolicy", env, verbose=1,
    learning_rate=3e-4, n_steps=256, batch_size=128, n_epochs=4,
    gamma=0.995, gae_lambda=0.95, ent_coef=0.08,
    clip_range=0.2, max_grad_norm=0.5,
    policy_kwargs=get_policy_kwargs(), device=DEVICE,
)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_dir = f"log/v2_10x10_{ts}"
model_10.set_logger(configure(log_dir, ["stdout", "csv", "tensorboard"]))

checkpoint_cb = CheckpointCallback(save_freq=500_000 // 16, save_path="checkpoints/",
                                    name_prefix="10x10")

model_10.learn(total_timesteps=TIMESTEPS, callback=[checkpoint_cb, ProgressCallback()])
model_10x10 = f"data/v2_10x10_{ts}"
model_10.save(model_10x10)
env.close()
save_to_drive(model_10x10, "v2_10x10_final")
print(f"\n10x10 model saved: {model_10x10}.zip")

In [ ]:
# 8. Test 10x10
RESULTS["10x10"] = test_model(model_10x10, 10, 12)
print(f"Results so far: {json.dumps({k: {kk: round(vv,1) for kk,vv in v.items()} for k,v in RESULTS.items()}, indent=2)}")
# Save intermediate results to Drive
if DRIVE_DIR:
    with open(f"{DRIVE_DIR}/results_partial.json", "w") as f:
        json.dump(RESULTS, f, indent=2)
    print(f"Partial results saved to Drive.")

In [ ]:
# 9. STAGE 3: Curriculum to 20x20 from 10x10 (3M steps, ~2.5h)
#    Lower LR + higher entropy for curriculum transfer to huge grid.
print("="*60)
print("STAGE 3: Curriculum 20x20 from 10x10 (3M steps)")
print("="*60)

DIM, OBS, MAX_STEPS, TIMESTEPS = 20, 48, 2000, 3_000_000

env = create_env(DIM, OBS, MAX_STEPS, n_envs=16)
model_20 = RecurrentPPO.load(model_10x10, env=env, device=DEVICE)
model_20.learning_rate = 1e-4
model_20.ent_coef = 0.1

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_dir = f"log/v2_20x20_{ts}"
model_20.set_logger(configure(log_dir, ["stdout", "csv", "tensorboard"]))

checkpoint_cb = CheckpointCallback(save_freq=500_000 // 16, save_path="checkpoints/",
                                    name_prefix="20x20")

model_20.learn(total_timesteps=TIMESTEPS, reset_num_timesteps=True,
               callback=[checkpoint_cb, ProgressCallback()])
model_20x20 = f"data/v2_20x20_{ts}"
model_20.save(model_20x20)
env.close()
save_to_drive(model_20x20, "v2_20x20_final")
print(f"\n20x20 model saved: {model_20x20}.zip")

In [ ]:
# 10. Test 20x20
RESULTS["20x20"] = test_model(model_20x20, 20, 48)
print(f"All results: {json.dumps({k: {kk: round(vv,1) for kk,vv in v.items()} for k,v in RESULTS.items()}, indent=2)}")

In [ ]:
# 11. Final Summary + Save
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
for name, r in RESULTS.items():
    print(f"  {name}: Coverage Rate={r['rate']:.1f}%, Avg Coverage={r['avg_cov']:.1f}%, Avg Steps={r['avg_steps']:.0f}")
print("="*60)

# Save full results
results_path = "data/v2_results.json"
with open(results_path, "w") as f:
    json.dump(RESULTS, f, indent=2)
print(f"\nResults saved to {results_path}")

if DRIVE_DIR:
    import shutil
    shutil.copy2(results_path, f"{DRIVE_DIR}/v2_results.json")
    print(f"Results saved to Drive: {DRIVE_DIR}/v2_results.json")

In [ ]:
# 12. Download all models
from google.colab import files
import zipfile

with zipfile.ZipFile("trained_models_v2.zip", "w") as zf:
    for f in [f"{model_5x5}.zip", f"{model_10x10}.zip", f"{model_20x20}.zip", results_path]:
        if os.path.exists(f):
            zf.write(f)
            print(f"  Added: {f}")
files.download("trained_models_v2.zip")
print("\nAll models downloaded!")